In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("ionosphere.csv")  

# Convert categorical labels to numerical
if df.iloc[:, -1].dtype == 'O':  
    df.iloc[:, -1] = LabelEncoder().fit_transform(df.iloc[:, -1])





# Split 
X = df.iloc[:, :-1].values  
y = df.iloc[:, -1].values  

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("Unique classes in y:", np.unique(y))

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


Unique classes in y: [0 1]


In [4]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def gini_impurity(y):
   
    classes = np.unique(y)
    impurity = 1 - sum((np.sum(y == c) / len(y))**2 for c in classes)
    return impurity

def split_data(X, y, feature, threshold):
    
    left_idx = X[:, feature] <= threshold
    right_idx = X[:, feature] > threshold
    return X[left_idx], X[right_idx], y[left_idx], y[right_idx]

def best_split(X, y):
    best_feature, best_threshold, best_score = None, None, float('inf')
    
    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            X_left, X_right, y_left, y_right = split_data(X, y, feature, threshold)
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            score = (len(y_left) * gini_impurity(y_left) + len(y_right) * gini_impurity(y_right)) / len(y)
            if score < best_score:
                best_feature, best_threshold, best_score = feature, threshold, score
                
    return best_feature, best_threshold

def build_tree(X, y, depth=0, max_depth=10):
    if len(set(y)) == 1 or depth >= max_depth:
        return Node(value=max(set(y), key=list(y).count))
    
    feature, threshold = best_split(X, y)
    if feature is None:
        return Node(value=max(set(y), key=list(y).count))

    X_left, X_right, y_left, y_right = split_data(X, y, feature, threshold)
    left_child = build_tree(X_left, y_left, depth + 1, max_depth)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth)

    return Node(feature=feature, threshold=threshold, left=left_child, right=right_child)

def predict_tree(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature] <= node.threshold:
        return predict_tree(node.left, x)
    else:
        return predict_tree(node.right, x)


In [5]:
class RandomForest:
    def __init__(self, n_trees=20, max_depth=10, sample_size=0.8):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.sample_size = sample_size
        self.trees = []

    def fit(self, X, y):
        """Train multiple decision trees on different subsets of data."""
        n_samples = int(self.sample_size * len(X))
        for _ in range(self.n_trees):
            idxs = np.random.choice(len(X), n_samples, replace=True)
            X_sample, y_sample = X[idxs], y[idxs]
            tree = build_tree(X_sample, y_sample, max_depth=self.max_depth)
            self.trees.append(tree)

    def predict(self, X):
        """Predict using majority voting across all trees."""
        predictions = np.array([[predict_tree(tree, x) for tree in self.trees] for x in X])
        return np.array([np.bincount(p).argmax() for p in predictions])

# Train Random Forest
rf = RandomForest(n_trees=20, max_depth=10)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)


In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

print("🔹 Accuracy:", accuracy_score(y_test, y_pred))
print("🔹 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("🔹 Precision:", precision_score(y_test, y_pred))
print("🔹 Recall:", recall_score(y_test, y_pred))
print("🔹 F1-Score:", f1_score(y_test, y_pred))
print("🔹 AUC Score:", roc_auc_score(y_test, y_pred))


🔹 Accuracy: 0.9238095238095239
🔹 Confusion Matrix:
 [[33  4]
 [ 4 64]]
🔹 Precision: 0.9411764705882353
🔹 Recall: 0.9411764705882353
🔹 F1-Score: 0.9411764705882353
🔹 AUC Score: 0.9165341812400636


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train sklearn's RandomForest
sklearn_rf = RandomForestClassifier(n_estimators=20, max_depth=10, random_state=42)
sklearn_rf.fit(X_train, y_train)
y_pred_sklearn = sklearn_rf.predict(X_test)

print("Accuracy (Implemented):", accuracy_score(y_test, y_pred))
print("Accuracy (Scikit-Learn):", accuracy_score(y_test, y_pred_sklearn))


Accuracy (Implemented): 0.9238095238095239
Accuracy (Scikit-Learn): 0.9428571428571428
